# Assignment 5 Resilient Colab Scaffold

This notebook is a setup and validation companion for the Assignment 5 scaffold.
It clones or refreshes the repo, mounts Google Drive, initializes W&B, prepares
local and Drive-backed output paths, and includes smoke-check cells you can run
in Colab after you finish the implementation.


In [24]:
from pathlib import Path

REPO_URL = "https://github.com/lllcr6/cs336-a5.git"
REPO_DIR = Path("/content/assignment5-alignment")
BRANCH = "main"

PYTHON_VERSION = "3.12"
VENV_DIR = REPO_DIR / ".venv"

WANDB_PROJECT = "cs336-assignment5-alignment"
WANDB_ENTITY = "chenrui6"
RUN_NAME = None

DRIVE_ROOT = Path("/content/drive/MyDrive/cs336-assignment5")
RUN_SUBDIR = "resilient-run"

SAVE_EVERY_STEPS = 10
EVAL_EVERY_STEPS = 10
RESUME_FROM_CHECKPOINT = None

LOCAL_OUTPUT_DIR = REPO_DIR / "outputs"
LOCAL_CHECKPOINT_DIR = LOCAL_OUTPUT_DIR / "checkpoints"
LOCAL_EVAL_DIR = LOCAL_OUTPUT_DIR / "eval"
LOCAL_WANDB_DIR = LOCAL_OUTPUT_DIR / "wandb"

DRIVE_RUN_DIR = DRIVE_ROOT / RUN_SUBDIR
DRIVE_CHECKPOINT_DIR = DRIVE_RUN_DIR / "checkpoints"
DRIVE_EVAL_DIR = DRIVE_RUN_DIR / "eval"
DRIVE_WANDB_DIR = DRIVE_RUN_DIR / "wandb"

In [25]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
import os
import subprocess

def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    return result

if REPO_DIR.exists():
    print(f'Refreshing existing repo at {REPO_DIR}')
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
    run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])
else:
    print(f'Cloning repo into {REPO_DIR}')
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
    run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

run(['git', '-C', str(REPO_DIR), 'status', '--short'])
run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

Refreshing existing repo at /content/assignment5-alignment
+ git -C /content/assignment5-alignment fetch origin

+ git -C /content/assignment5-alignment checkout main
Already on 'main'
Your branch is up to date with 'origin/main'.

+ git -C /content/assignment5-alignment pull --ff-only origin main
From https://github.com/lllcr6/cs336-a5
 * branch            main       -> FETCH_HEAD
Already up to date.

Working directory: /content/assignment5-alignment
+ git -C /content/assignment5-alignment status --short
?? outputs/

+ git -C /content/assignment5-alignment log -1 --oneline
aa0eb3b implement masked_mean



CompletedProcess(args=['git', '-C', '/content/assignment5-alignment', 'log', '-1', '--oneline'], returncode=0, stdout='aa0eb3b implement masked_mean\n')

In [32]:
import os
import sys
import shlex
import site
import subprocess
from pathlib import Path

def bash(cmd, cwd=None, check=True):
    print("+", cmd)
    subprocess.run(["bash", "-lc", cmd], cwd=cwd, check=check)

bash(f"{sys.executable} -m pip install -U uv")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv python install {PYTHON_VERSION}")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv venv --python {PYTHON_VERSION} .venv")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv sync --no-install-package flash-attn")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv sync")

print("uv environment ready at:", VENV_DIR)
print("venv exists:", VENV_DIR.exists())

+ /usr/bin/python3 -m pip install -U uv
+ cd /content/assignment5-alignment && uv python install 3.12
+ cd /content/assignment5-alignment && uv venv --python 3.12 .venv
+ cd /content/assignment5-alignment && uv sync --no-install-package flash-attn
+ cd /content/assignment5-alignment && uv sync
uv environment ready at: /content/assignment5-alignment/.venv
venv exists: True


In [33]:
for path in [
    LOCAL_OUTPUT_DIR,
    LOCAL_CHECKPOINT_DIR,
    LOCAL_EVAL_DIR,
    LOCAL_WANDB_DIR,
    DRIVE_RUN_DIR,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_EVAL_DIR,
    DRIVE_WANDB_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)
    print('created', path)


created /content/assignment5-alignment/outputs
created /content/assignment5-alignment/outputs/checkpoints
created /content/assignment5-alignment/outputs/eval
created /content/assignment5-alignment/outputs/wandb
created /content/drive/MyDrive/cs336-assignment5/resilient-run
created /content/drive/MyDrive/cs336-assignment5/resilient-run/checkpoints
created /content/drive/MyDrive/cs336-assignment5/resilient-run/eval
created /content/drive/MyDrive/cs336-assignment5/resilient-run/wandb


In [34]:
import wandb

wandb.login()
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    dir=str(LOCAL_WANDB_DIR),
    tags=['assignment5', 'colab', 'resilient'],
    resume='allow',
)

print('W&B run:', wandb_run.name if wandb_run else None)


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B run: effortless-dawn-2


## Persistence And Resume

Future training code should save checkpoints every `SAVE_EVERY_STEPS`, write W&B
metrics continuously, and mirror new checkpoints into Google Drive after each
save event. If Colab disconnects, you can resume from an explicit checkpoint or
from the newest checkpoint discovered in Drive.


In [35]:

from cs336_alignment.checkpointing import (
    latest_checkpoint,
    resolve_resume_checkpoint,
    sync_checkpoint_to_drive,
)

print('latest drive checkpoint:', latest_checkpoint(DRIVE_CHECKPOINT_DIR))
print('resume candidate:', resolve_resume_checkpoint(
    explicit_resume_path=RESUME_FROM_CHECKPOINT,
    drive_checkpoint_dir=DRIVE_CHECKPOINT_DIR,
    local_checkpoint_dir=LOCAL_CHECKPOINT_DIR,
))
print('drive sync helper:', sync_checkpoint_to_drive)


latest drive checkpoint: None
resume candidate: None


In [36]:
from cs336_alignment.config import (
    CheckpointConfig,
    DriveSyncConfig,
    EvalConfig,
    ExpertIterationConfig,
    GRPOConfig,
    SFTConfig,
    WandbConfig,
)

wandb_config = WandbConfig(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    run_name=RUN_NAME,
    tags=['assignment5', 'colab', 'resilient'],
    log_dir=LOCAL_WANDB_DIR,
)
checkpoint_config = CheckpointConfig(
    output_dir=LOCAL_CHECKPOINT_DIR,
    save_every_steps=SAVE_EVERY_STEPS,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)
drive_sync_config = DriveSyncConfig(
    enabled=True,
    drive_root=DRIVE_ROOT,
    per_run_subdir=RUN_SUBDIR,
)
eval_config = EvalConfig(
    output_dir=LOCAL_EVAL_DIR,
    batch_size=1,
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop_tokens=['</answer>'],
)
sft_config = SFTConfig(
    train_steps=0,
    gradient_accumulation_steps=1,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)
grpo_config = GRPOConfig(
    train_steps=0,
    rollout_batch_size=0,
    train_batch_size=0,
    gradient_accumulation_steps=1,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)
expert_iteration_config = ExpertIterationConfig(
    n_ei_steps=0,
    rollout_batch_size=0,
    rollouts_per_example=0,
    sft_epochs_per_round=0,
    eval_every_steps=EVAL_EVERY_STEPS,
    save_every_steps=SAVE_EVERY_STEPS,
    wandb=wandb_config,
    checkpoint=checkpoint_config,
    drive_sync=drive_sync_config,
)

print(wandb_config)
print(checkpoint_config)
print(drive_sync_config)


WandbConfig(project='cs336-assignment5-alignment', entity='chenrui6', run_name=None, tags=['assignment5', 'colab', 'resilient'], log_dir=PosixPath('/content/assignment5-alignment/outputs/wandb'))
CheckpointConfig(output_dir=PosixPath('/content/assignment5-alignment/outputs/checkpoints'), save_every_steps=10, max_checkpoints=3, resume_from_checkpoint=None)
DriveSyncConfig(enabled=True, drive_root=PosixPath('/content/drive/MyDrive/cs336-assignment5'), per_run_subdir='resilient-run', checkpoint_dirname='checkpoints', log_dirname='logs')


In [51]:
import cs336_alignment
import cs336_alignment.config as config_module
import cs336_alignment.sft as sft_module
import cs336_alignment.grpo as grpo_module
import cs336_alignment.evaluation as evaluation_module
import tests.adapters as adapters_module

print('cs336_alignment:', cs336_alignment)
print('config:', config_module)
print('sft:', sft_module)
print('grpo:', grpo_module)
print('evaluation:', evaluation_module)
print('adapters:', adapters_module)


cs336_alignment: <module 'cs336_alignment' from '/content/assignment5-alignment/cs336_alignment/__init__.py'>
config: <module 'cs336_alignment.config' from '/content/assignment5-alignment/cs336_alignment/config.py'>
sft: <module 'cs336_alignment.sft' from '/content/assignment5-alignment/cs336_alignment/sft.py'>
grpo: <module 'cs336_alignment.grpo' from '/content/assignment5-alignment/cs336_alignment/grpo.py'>
evaluation: <module 'cs336_alignment.evaluation' from '/content/assignment5-alignment/cs336_alignment/evaluation.py'>
adapters: <module 'tests.adapters' from '/content/assignment5-alignment/tests/adapters.py'>


In [52]:
from cs336_alignment import (
    run_zero_shot_baseline,
    run_sft_training,
    run_expert_iteration,
    run_grpo_training,
)

print(run_zero_shot_baseline)
print(run_sft_training)
print(run_expert_iteration)
print(run_grpo_training)


<function run_zero_shot_baseline at 0x7966b2674680>
<function run_sft_training at 0x7966b3a9e5c0>
<function run_expert_iteration at 0x7966b3a9ef20>
<function run_grpo_training at 0x7966b26744a0>


## What To Run In Colab

Once the scaffold is implemented, use the cells above to confirm imports, Drive
setup, and checkpoint discovery. The package now owns checkpoint discovery,
Drive mirroring, and resume selection, so the notebook stays lightweight while
still matching the Colab workflow.
